# Pendle Finance TVL Collapse: Sep 2025 – Apr 2026
## Comprehensive Research Analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

BASE = "."  # run from project root

# Create figures directory
os.makedirs(os.path.join(BASE, 'figures'), exist_ok=True)

print('Setup complete. Figures will be saved to ./figures/')

---
## Section 1 — Wave 1: September 2025 Structural Expiry

In [ ]:
# ── Figure 1: sUSDe Supply Aug–Oct 2025 ──────────────────────────────────────
susde = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/susde_supply_aug_oct_2025.csv')
)
susde['date'] = pd.to_datetime(susde['date'])
susde = susde.sort_values('date')

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(susde['date'], susde['sUSDe_supply'] / 1e9,
        color='#2196F3', linewidth=2.5, marker='o', markersize=5, label='sUSDe Supply')
ax.fill_between(susde['date'], susde['sUSDe_supply'] / 1e9, alpha=0.15, color='#2196F3')

sep_expiry = pd.Timestamp('2025-09-25')
ax.axvline(sep_expiry, color='#E53935', linestyle='--', linewidth=1.8, label='Sep-25 Expiry')

peak_row = susde.loc[susde['sUSDe_supply'].idxmax()]
ax.annotate(
    f"Peak: {peak_row['sUSDe_supply']/1e9:.2f}B\n({peak_row['date'].strftime('%b %d')})",
    xy=(peak_row['date'], peak_row['sUSDe_supply'] / 1e9),
    xytext=(20, -35), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='#333'),
    fontsize=10, color='#333'
)

ax.set_title('Fig 1: sUSDe Total Supply — Aug to Oct 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Supply (Billions USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30)
ax.legend(loc='upper left')
ax.text(0.02, 0.05,
        'Key finding: sUSDe supply peaked ~$4.9B before Sep-25 expiry, then contracted sharply as PT redemptions unwound',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig01_susde_supply.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 1 saved.')

In [ ]:
# ── Figure 2: Aave PT-sUSDE and PT-USDe Collateral Sep 22–30 ─────────────────
aave = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/aave_pt_susde_sep2025.csv')
)
aave['date'] = pd.to_datetime(aave['date'])

pt_assets = ['PT-sUSDE-25SEP2025', 'PT-USDe-25SEP2025']
aave_filt = aave[aave['asset'].isin(pt_assets)].copy()
aave_filt = aave_filt[(aave_filt['date'] >= '2025-09-22') & (aave_filt['date'] <= '2025-09-30')]
aave_pivot = aave_filt.pivot_table(index='date', columns='asset', values='totalAToken_units', aggfunc='sum')
aave_pivot = aave_pivot / 1e9  # convert to billions

fig, ax = plt.subplots(figsize=(12, 5))

dates = aave_pivot.index
x = np.arange(len(dates))
width = 0.38

col_colors = ['#1565C0', '#42A5F5']
for i, col in enumerate(aave_pivot.columns):
    ax.bar(x + (i - 0.5) * width, aave_pivot[col].values,
           width=width, label=col, color=col_colors[i], alpha=0.85)

# Annotate peak combined
combined = aave_pivot.sum(axis=1)
peak_idx_label = combined.idxmax()
peak_x_pos = list(dates).index(peak_idx_label)
ax.annotate(
    '$3.09B peak on Sep-24',
    xy=(peak_x_pos, combined.max()),
    xytext=(peak_x_pos + 0.5, combined.max() + 0.08),
    arrowprops=dict(arrowstyle='->', color='#B71C1C'),
    fontsize=10, color='#B71C1C', fontweight='bold'
)

ax.set_title('Fig 2: Aave PT-sUSDE & PT-USDe Collateral Sep 22–30, 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Collateral (Billions USD)')
ax.set_xticks(x)
ax.set_xticklabels([d.strftime('%b-%d') for d in dates], rotation=30)
ax.legend(loc='upper right')
ax.text(0.02, 0.05,
        'Key finding: ~$3.09B in PT collateral peaked on Sep-24, then collapsed ~96% within 2 days of expiry (Sep-25)',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig02_aave_pt_collateral.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 2 saved.')

In [ ]:
# ── Figure 3: Per-Pool TVL — Daily Net Flow Stacked Bar Sep 1–Oct 14 ─────────
pool_tvl = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/per_pool_tvl.csv')
)
pool_tvl['day'] = pd.to_datetime(pool_tvl['day'])
pool_tvl = pool_tvl[(pool_tvl['day'] >= '2025-09-01') & (pool_tvl['day'] <= '2025-10-14')]

pool_pivot = pool_tvl.pivot_table(index='day', columns='pool_name', values='net_flow', aggfunc='sum')
pool_pivot = pool_pivot.fillna(0) / 1e9
pool_pivot = pool_pivot.sort_index()

pool_colors = {
    'PT-USDe-25SEP2025': '#E53935',
    'PT-sUSDE-25SEP2025': '#FB8C00',
    'PT-USDe-27NOV2025': '#43A047',
    'PT-sUSDE-27NOV2025': '#1E88E5',
}

fig, ax = plt.subplots(figsize=(14, 6))

dates = pool_pivot.index
x = np.arange(len(dates))
width = 0.75

bottom_pos = np.zeros(len(dates))
bottom_neg = np.zeros(len(dates))
for col in pool_pivot.columns:
    vals = pool_pivot[col].values
    color = pool_colors.get(col, '#888')
    pos_vals = np.where(vals > 0, vals, 0)
    neg_vals = np.where(vals < 0, vals, 0)
    ax.bar(x, pos_vals, width=width, bottom=bottom_pos, color=color, label=col, alpha=0.85)
    ax.bar(x, neg_vals, width=width, bottom=bottom_neg, color=color, alpha=0.85)
    bottom_pos = bottom_pos + pos_vals
    bottom_neg = bottom_neg + neg_vals

ax.axhline(0, color='black', linewidth=0.8)

sep25_ts = pd.Timestamp('2025-09-25')
dates_list = list(dates)
if sep25_ts in dates_list:
    sep_idx = dates_list.index(sep25_ts)
    ax.axvline(sep_idx, color='#B71C1C', linestyle='--', linewidth=2, label='Sep-25 Expiry')

tick_step = max(1, len(dates) // 15)
ax.set_xticks(x[::tick_step])
ax.set_xticklabels([d.strftime('%b-%d') for d in dates[::tick_step]], rotation=40, ha='right')
ax.set_title('Fig 3: Per-Pool Daily Net Flow (Sep 1 – Oct 14, 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Net Flow (Billions USD)')
ax.legend(loc='upper left', fontsize=9)
ax.text(0.02, 0.05,
        'Key finding: Sep-25 pools show massive negative flow at expiry; Nov-27 pools absorbed some inflows but not the full volume',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig03_per_pool_net_flow.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 3 saved.')

---
## Section 2 — Wave 2: Trust Collapse Q4 2025

In [ ]:
# ── Figure 4: PT Implied Yield vs Aave USDC Borrow Rate ──────────────────────
yields_df = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/pt_implied_yields_sep_dec_2025.csv')
)
yields_df['date'] = pd.to_datetime(yields_df['date'])
yields_df = yields_df[yields_df['implied_APY_%'].notna()]

# Filter to NOV pools only (Sep pools expired)
nov_pools = ['PT-sUSDE-27NOV2025', 'PT-USDe-27NOV2025']
nov_yields = yields_df[yields_df['pool'].isin(nov_pools)].copy()

# Group by date and pool taking mean
nov_weekly = nov_yields.groupby(['date', 'pool'])['implied_APY_%'].mean().reset_index()

# Aave USDC borrow rate — Alchemy RPC rows only
aave_borrow = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/aave_usdc_borrow_rate_sep_dec_2025.csv')
)
aave_borrow['date'] = pd.to_datetime(aave_borrow['date'])
aave_rpc = aave_borrow[aave_borrow['source'] == 'Alchemy_RPC'][['date', 'variableBorrowAPR_%']].dropna()
aave_rpc = aave_rpc.sort_values('date')

fig, ax = plt.subplots(figsize=(13, 6))

pool_line_colors = {
    'PT-sUSDE-27NOV2025': '#1565C0',
    'PT-USDe-27NOV2025': '#00897B',
}
for pool, grp in nov_weekly.groupby('pool'):
    grp = grp.sort_values('date')
    ax.plot(grp['date'], grp['implied_APY_%'],
            color=pool_line_colors.get(pool, '#888'),
            linewidth=2.2, marker='o', markersize=4.5, label=pool)

ax.plot(aave_rpc['date'], aave_rpc['variableBorrowAPR_%'],
        color='#E53935', linewidth=2, linestyle='--', label='Aave USDC Borrow Rate (RPC)')

# Shade carry-inverted region using interpolation
date_range = pd.date_range(aave_rpc['date'].min(), aave_rpc['date'].max(), freq='D')
borrow_interp = np.interp(
    mdates.date2num(date_range),
    mdates.date2num(aave_rpc['date']),
    aave_rpc['variableBorrowAPR_%'].values
)
min_yield_by_date = nov_weekly.groupby('date')['implied_APY_%'].min().reset_index().sort_values('date')
min_yield_interp = np.interp(
    mdates.date2num(date_range),
    mdates.date2num(min_yield_by_date['date']),
    min_yield_by_date['implied_APY_%'].values
)
inverted_mask = borrow_interp > min_yield_interp
ax.fill_between(date_range, borrow_interp, min_yield_interp,
                where=inverted_mask, alpha=0.18, color='#E53935', label='Carry Inverted Zone')

sep_expiry = pd.Timestamp('2025-09-25')
ax.axvline(sep_expiry, color='#6A1B9A', linestyle='--', linewidth=1.5, label='Sep-25 Expiry')

ax.set_title('Fig 4: PT Implied APY vs Aave USDC Borrow Rate (Sep–Dec 2025)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('APY / Borrow Rate (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.xticks(rotation=30)
ax.legend(loc='upper right', fontsize=9)
ax.text(0.02, 0.05,
        'Key finding: PT yields compressed from ~13% to <5% by Dec-2025; borrow rates exceeded implied yields (carry inversion) in late Nov',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig04_pt_yield_vs_borrow.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 4 saved.')

In [ ]:
# ── Figure 5: DAU Q4 2025 with 7d Moving Average ─────────────────────────────
dau_q4 = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/dau_q4_2025.csv')
)
dau_q4['day'] = pd.to_datetime(dau_q4['day'])
dau_q4 = dau_q4[(dau_q4['day'] >= '2025-10-01') & (dau_q4['day'] <= '2025-12-31')]
dau_q4 = dau_q4.sort_values('day')

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(dau_q4['day'], dau_q4['dau'], color='#1976D2', alpha=0.55, width=0.9, label='Daily Active Users')
ax.plot(dau_q4['day'], dau_q4['dau_7d_ma'],
        color='#E53935', linewidth=2.2, label='7-Day MA')

# Mark Nov-6 peak
nov6 = pd.Timestamp('2025-11-06')
nov6_row = dau_q4[dau_q4['day'] == nov6]
if not nov6_row.empty:
    dau_peak_val = nov6_row['dau'].values[0]
    ax.annotate(
        f'Peak: {int(dau_peak_val):,} users\n(Nov-6)',
        xy=(nov6, dau_peak_val),
        xytext=(20, 10), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#B71C1C'),
        fontsize=10, color='#B71C1C', fontweight='bold'
    )

ax.set_title('Fig 5: Daily Active Users — Q4 2025 (Oct–Dec)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Active Users')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.xticks(rotation=30)
ax.legend(loc='upper left')
ax.text(0.02, 0.05,
        'Key finding: DAU peaked at 8,874 on Nov-6 (likely Nov-27 expiry speculation), then declined ~70% through Dec-2025',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig05_dau_q4_2025.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 5 saved.')

In [ ]:
# ── Figure 6: vePENDLE Lock Rate Oct–Dec 2025 ─────────────────────────────────
vependle = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/vependle_lock_rate_oct_dec_2025.csv')
)
vependle['date'] = pd.to_datetime(vependle['date'])
vependle = vependle.sort_values('date')

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(vependle['date'], vependle['lock_rate_%'],
        color='#6A1B9A', linewidth=2.5, marker='o', markersize=6, label='vePENDLE Lock Rate')
ax.fill_between(vependle['date'], vependle['lock_rate_%'], alpha=0.15, color='#6A1B9A')

# Curve benchmark reference line at 50%
ax.axhline(50, color='#F57F17', linestyle='--', linewidth=1.8, label='Curve ~50% Benchmark')

# Annotate peak
peak_row = vependle.loc[vependle['lock_rate_%'].idxmax()]
ax.annotate(
    f"~{peak_row['lock_rate_%']:.1f}% peak\n({peak_row['date'].strftime('%b-%d')})",
    xy=(peak_row['date'], peak_row['lock_rate_%']),
    xytext=(15, 10), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='#4A148C'),
    fontsize=10, color='#4A148C', fontweight='bold'
)

ax.set_title('Fig 6: vePENDLE Lock Rate — Oct to Dec 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Lock Rate (%)')
ax.set_ylim(0, 60)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.xticks(rotation=30)
ax.legend(loc='upper left')
ax.text(0.02, 0.05,
        f"Key finding: Lock rate peaked at ~{peak_row['lock_rate_%']:.1f}% (Dec-2025) — well below Curve's ~50% benchmark, indicating weaker governance commitment",
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig06_vependle_lock_rate.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 6 saved.')

---
## Section 3 — Wave 3: Slow Bleed Q1 2026

In [ ]:
# ── Figure 7: Indexed Price Performance (Base = Nov 1, 2025 = 100) ────────────
prices = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave3_slow_bleed/data/prices_jan_apr_2026.csv')
)
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values('date').reset_index(drop=True)

# Base = Nov 1, 2025
base_date = pd.Timestamp('2025-11-01')
base_mask = prices['date'] == base_date
if base_mask.any():
    base_row = prices[base_mask].iloc[0]
else:
    # fallback: first row
    base_row = prices.iloc[0]

prices['BTC_idx'] = prices['BTC'] / base_row['BTC'] * 100
prices['ETH_idx'] = prices['ETH'] / base_row['ETH'] * 100
prices['PENDLE_idx'] = prices['PENDLE'] / base_row['PENDLE'] * 100

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(prices['date'], prices['BTC_idx'], color='#F57F17', linewidth=2.3, label='BTC')
ax.plot(prices['date'], prices['ETH_idx'], color='#1565C0', linewidth=2.3, label='ETH')
ax.plot(prices['date'], prices['PENDLE_idx'], color='#2E7D32', linewidth=2.3, label='PENDLE')
ax.axhline(100, color='#9E9E9E', linestyle='--', linewidth=1.3, label='Base (Nov 1, 2025 = 100)')

# Key event markers
ymin_disp = prices[['PENDLE_idx']].min().min()
ymax_disp = prices[['BTC_idx', 'ETH_idx', 'PENDLE_idx']].max().max()
events = [
    ('2026-01-20', 'sPENDLE\nLaunch'),
    ('2026-01-29', 'vePENDLE\nRenewals\nCease'),
    ('2026-02-02', 'AIM\nLaunch'),
]
text_y_positions = [ymin_disp + (ymax_disp - ymin_disp) * f for f in [0.55, 0.40, 0.25]]
for (evdate, label), ty in zip(events, text_y_positions):
    ev = pd.Timestamp(evdate)
    ax.axvline(ev, color='#E53935', linestyle=':', linewidth=1.5, alpha=0.8)
    ax.text(ev, ty, label, fontsize=8, color='#C62828', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.75))

pendle_drawdown = (prices['PENDLE_idx'].iloc[-1] - 100)
ax.set_title('Fig 7: Indexed Price Performance (Base = Nov 1, 2025 = 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Indexed Price (Nov 1 = 100)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
ax.legend(loc='upper right')
ax.text(0.02, 0.05,
        f'Key finding: PENDLE declined ~{abs(pendle_drawdown):.0f} index pts from base while BTC rose; severe underperformance through Q1 2026',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig07_indexed_prices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 7 saved.')

In [ ]:
# ── Figure 8: Full DAU Timeline Oct 2025–Apr 2026 ─────────────────────────────
dau_q1 = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave3_slow_bleed/data/dau_q1_2026.csv')
)
dau_q1['day'] = pd.to_datetime(dau_q1['day'])
dau_q1 = dau_q1.sort_values('day')

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(dau_q1['day'], dau_q1['dau'], color='#1976D2', alpha=0.5, width=0.9, label='Daily Active Users')
ax.plot(dau_q1['day'], dau_q1['dau_30d_ma'],
        color='#E53935', linewidth=2.3, label='30-Day MA')

# Floor zone shade (350–600)
ax.axhspan(350, 600, alpha=0.12, color='#2E7D32', label='Floor Zone (350–600 users)')

# Mark Nov-6 peak
nov6 = pd.Timestamp('2025-11-06')
nov6_row = dau_q1[dau_q1['day'] == nov6]
if not nov6_row.empty:
    ax.annotate(
        'Peak: 8,874\n(Nov-6)',
        xy=(nov6, nov6_row['dau'].values[0]),
        xytext=(30, 5), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#B71C1C'),
        fontsize=10, color='#B71C1C', fontweight='bold'
    )

# Annotate Apr-6
apr6 = pd.Timestamp('2026-04-06')
apr6_row = dau_q1[dau_q1['day'] == apr6]
if not apr6_row.empty:
    dau_val = apr6_row['dau'].values[0]
    ax.annotate(
        f'Apr-6: {int(dau_val)} users',
        xy=(apr6, dau_val),
        xytext=(-60, 30), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#555'),
        fontsize=10, color='#555'
    )

ax.set_title('Fig 8: Full DAU Timeline — Oct 2025 to Apr 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Active Users')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
ax.legend(loc='upper right')
ax.text(0.02, 0.05,
        'Key finding: DAU collapsed 97%+ from the Nov-6 peak of 8,874 to 214 by Apr-6, settling in the 350–600 floor zone',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig08_dau_full_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 8 saved.')

In [ ]:
# ── Figure 9: vePENDLE vs sPENDLE Supply Jan–Apr 2026 ─────────────────────────
spendle = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave3_slow_bleed/data/spendle_migration_jan_apr_2026.csv')
)
spendle['date'] = pd.to_datetime(spendle['date'])
spendle = spendle.sort_values('date')
spendle_filt = spendle[spendle['date'] >= '2026-01-01'].copy()

fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

# vePENDLE on left axis (declining)
l1, = ax1.plot(spendle_filt['date'], spendle_filt['vePENDLE_supply'] / 1e6,
               color='#6A1B9A', linewidth=2.5, marker='o', markersize=5, label='vePENDLE Supply (M)')
ax1.fill_between(spendle_filt['date'], spendle_filt['vePENDLE_supply'] / 1e6,
                 alpha=0.12, color='#6A1B9A')

# sPENDLE on right axis (growing) — only where not NaN
sp_valid = spendle_filt[spendle_filt['sPENDLE_supply'].notna()].copy()
l2, = ax2.plot(sp_valid['date'], sp_valid['sPENDLE_supply'] / 1e6,
               color='#00897B', linewidth=2.5, marker='s', markersize=5,
               label='sPENDLE Supply (M)')
ax2.fill_between(sp_valid['date'], sp_valid['sPENDLE_supply'] / 1e6,
                 alpha=0.12, color='#00897B')

# Event markers
jan20 = pd.Timestamp('2026-01-20')
jan29 = pd.Timestamp('2026-01-29')
ax1.axvline(jan20, color='#E53935', linestyle='--', linewidth=1.8)
ax1.axvline(jan29, color='#F57F17', linestyle='--', linewidth=1.8)

ve_max = spendle_filt['vePENDLE_supply'].max() / 1e6
ax1.text(jan20, ve_max * 1.01, 'Jan-20\nsPENDLE\nLaunch',
         fontsize=9, color='#C62828', ha='center',
         bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
ax1.text(jan29, ve_max * 0.97, 'Jan-29\nLast\nRenewal',
         fontsize=9, color='#E65100', ha='center',
         bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax1.set_title('Fig 9: vePENDLE vs sPENDLE Supply — Jan to Apr 2026', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('vePENDLE Supply (Millions)', color='#6A1B9A')
ax2.set_ylabel('sPENDLE Supply (Millions)', color='#00897B')
ax1.tick_params(axis='y', labelcolor='#6A1B9A')
ax2.tick_params(axis='y', labelcolor='#00897B')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.xticks(rotation=30)
ax1.legend([l1, l2], [l1.get_label(), l2.get_label()], loc='center left')
ax1.text(0.02, 0.05,
         'Key finding: vePENDLE declined as sPENDLE migrated governance; sPENDLE adoption reached ~18.7% by Apr-6',
         transform=ax1.transAxes, fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig09_vependle_spendle_migration.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 9 saved.')

---
## Section 4 — Q2 User Behavior

In [ ]:
# ── Figure 10: YT/PT Volume Ratio ─────────────────────────────────────────────
yt_pt = pd.read_csv(
    os.path.join(BASE, 'q2_metrics/user_behavior/data/yt_pt_ratio.csv')
)
yt_pt['week'] = pd.to_datetime(yt_pt['week'])
yt_pt = yt_pt.sort_values('week').reset_index(drop=True)

fig = plt.figure(figsize=(14, 9))
gs = GridSpec(2, 1, figure=fig, hspace=0.45)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1])

# Top: stacked bar
pt_vol = yt_pt['pt_volume'] / 1e9
yt_vol = yt_pt['yt_volume'] / 1e9
x = np.arange(len(yt_pt))
width = 0.78
ax_top.bar(x, pt_vol, width=width, color='#1565C0', label='PT Volume', alpha=0.85)
ax_top.bar(x, yt_vol, width=width, bottom=pt_vol, color='#00897B', label='YT Volume', alpha=0.85)

# Mark Sep-25 and Nov-27 expiry
# After reset_index, idxmin returns a positional integer directly
sep25_x = int((yt_pt['week'] - pd.Timestamp('2025-09-22')).abs().idxmin())
nov27_x = int((yt_pt['week'] - pd.Timestamp('2025-11-24')).abs().idxmin())

ax_top.axvline(sep25_x + 0.5, color='#E53935', linestyle='--', linewidth=1.8, label='Sep-25 Expiry')
ax_top.axvline(nov27_x + 0.5, color='#F57F17', linestyle='--', linewidth=1.8, label='Nov-27 Expiry')

tick_step = max(1, len(yt_pt) // 12)
ax_top.set_xticks(x[::tick_step])
ax_top.set_xticklabels([d.strftime("%b-%d\n'%y") for d in yt_pt['week'].iloc[::tick_step]], fontsize=9)
ax_top.set_ylabel('Volume (Billions USD)')
ax_top.set_title('Fig 10: PT vs YT Volume by Week (Sep 2025–Apr 2026)', fontsize=14, fontweight='bold')
ax_top.legend(loc='upper right', fontsize=9)

# Bottom: YT/PT ratio line
ax_bot.plot(x, yt_pt['yt_pt_ratio'], color='#E53935', linewidth=2.2, marker='o', markersize=4)
ax_bot.axhline(1.0, color='#555', linestyle='--', linewidth=1.5, label='Parity (ratio=1)')
ax_bot.axvline(sep25_x + 0.5, color='#E53935', linestyle='--', linewidth=1.8)
ax_bot.axvline(nov27_x + 0.5, color='#F57F17', linestyle='--', linewidth=1.8)

ax_bot.set_xticks(x[::tick_step])
ax_bot.set_xticklabels([d.strftime("%b-%d\n'%y") for d in yt_pt['week'].iloc[::tick_step]], fontsize=9)
ax_bot.set_ylabel('YT/PT Ratio')
ax_bot.set_xlabel('Week')
ax_bot.legend(loc='upper right')
ax_bot.text(0.02, 0.08,
            'Key finding: YT/PT ratio collapsed post-Sep-25 expiry from ~0.8 to near 0, showing users abandoned speculative YT positions',
            transform=ax_bot.transAxes, fontsize=9, color='#555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig10_yt_pt_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 10 saved.')

In [ ]:
# ── Figure 11: Governance Stats — sPENDLE Adoption vs vePENDLE Lock Rate ──────
gov = pd.read_csv(
    os.path.join(BASE, 'q2_metrics/token_governance/data/governance_stats.csv')
)
gov['date'] = pd.to_datetime(gov['date'])
gov = gov.sort_values('date')

gov_lock = gov[gov['vePENDLE_lock_rate_%'].notna()]
gov_spendle = gov[gov['sPENDLE_adoption_%'].notna()]

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

l1, = ax1.plot(gov_lock['date'], gov_lock['vePENDLE_lock_rate_%'],
               color='#6A1B9A', linewidth=2.3, marker='o', markersize=7,
               label='vePENDLE Lock Rate (%)')
l2, = ax2.plot(gov_spendle['date'], gov_spendle['sPENDLE_adoption_%'],
               color='#00897B', linewidth=2.3, marker='s', markersize=7,
               label='sPENDLE Adoption (%)')

ax1.set_title('Fig 11: sPENDLE Adoption % vs vePENDLE Lock Rate Over Time',
              fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('vePENDLE Lock Rate (%)', color='#6A1B9A')
ax2.set_ylabel('sPENDLE Adoption (%)', color='#00897B')
ax1.tick_params(axis='y', labelcolor='#6A1B9A')
ax2.tick_params(axis='y', labelcolor='#00897B')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.xticks(rotation=30)
ax1.legend([l1, l2], [l1.get_label(), l2.get_label()], loc='center left')

# Annotate final sPENDLE adoption point
if not gov_spendle.empty:
    last = gov_spendle.iloc[-1]
    ax2.annotate(
        f"{last['sPENDLE_adoption_%']:.1f}% adoption\n({last['date'].strftime('%b-%d')})",
        xy=(last['date'], last['sPENDLE_adoption_%']),
        xytext=(-70, 15), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#00695C'),
        fontsize=10, color='#00695C', fontweight='bold'
    )

ax1.text(0.02, 0.05,
         'Key finding: sPENDLE adoption grew to ~18.7% by Apr-6 while vePENDLE lock rate stayed range-bound, indicating slow governance migration',
         transform=ax1.transAxes, fontsize=9, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig11_governance_stats.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 11 saved.')

---
## Section 5 — Q3 Case Studies

In [ ]:
# ── Figure 12: MakerDAO/Sky TVL History 2022–2025 ─────────────────────────────
maker = pd.read_csv(
    os.path.join(BASE, 'q3_recommendations/data/makerdao_sky_tvl.csv')
)
maker['date'] = pd.to_datetime(maker['date'])
maker = maker.sort_values('date')

# Use 'makerdao' protocol, filter from 2022
maker_filt = maker[(maker['protocol'] == 'makerdao') & (maker['date'] >= '2022-01-01')].copy()

fig, ax = plt.subplots(figsize=(13, 6))

ax.fill_between(maker_filt['date'], maker_filt['totalLiquidityUSD'] / 1e9,
                alpha=0.25, color='#1565C0')
ax.plot(maker_filt['date'], maker_filt['totalLiquidityUSD'] / 1e9,
        color='#1565C0', linewidth=2.2, label='MakerDAO/Sky TVL')

# ETH crash Jun 2022 annotation
eth_crash_date = pd.Timestamp('2022-06-15')
candidates = maker_filt[maker_filt['date'] >= eth_crash_date]
if not candidates.empty:
    crash_row = candidates.iloc[0]
    ax.annotate(
        'ETH crash\nJun 2022',
        xy=(crash_row['date'], crash_row['totalLiquidityUSD'] / 1e9),
        xytext=(30, 2), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#B71C1C'),
        fontsize=9, color='#B71C1C', fontweight='bold'
    )

# RWA >50% of backing ~2024 annotation
rwa_date = pd.Timestamp('2024-01-01')
rwa_candidates = maker_filt[maker_filt['date'] >= rwa_date]
if not rwa_candidates.empty:
    rwa_row = rwa_candidates.iloc[0]
    ax.annotate(
        'RWA >50%\nof backing (2024)',
        xy=(rwa_row['date'], rwa_row['totalLiquidityUSD'] / 1e9),
        xytext=(5, 35), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#1B5E20'),
        fontsize=9, color='#1B5E20', fontweight='bold'
    )

ax.set_title('Fig 12: MakerDAO/Sky TVL History (2022–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('TVL (Billions USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30)
ax.legend(loc='upper right')
ax.text(0.02, 0.05,
        'Key finding: MakerDAO recovered from 2022 ETH crash by diversifying into RWA — a diversification blueprint for Pendle',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig12_makerdao_tvl.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 12 saved.')

In [ ]:
# ── Figure 13: Curve TVL vs RWA Protocol TVL Growth ───────────────────────────
curve = pd.read_csv(
    os.path.join(BASE, 'q3_recommendations/data/curve_tvl.csv')
)
rwa = pd.read_csv(
    os.path.join(BASE, 'q3_recommendations/data/rwa_protocols_tvl.csv')
)
rwa['date'] = pd.to_datetime(rwa['date'])

# Curve: all dates are NaT; reconstruct synthetic daily dates.
# Dataset has 2250 rows; assign daily dates starting 2019-07-01
curve_start = pd.Timestamp('2019-07-01')
curve['date_synth'] = pd.date_range(start=curve_start, periods=len(curve), freq='D')
curve_filt = curve[curve['date_synth'] >= '2022-01-01'].copy()

# RWA: aggregate total TVL across all protocols by date
rwa_agg = rwa.groupby('date')['totalLiquidityUSD'].sum().reset_index().sort_values('date')
rwa_agg_filt = rwa_agg[rwa_agg['date'] >= '2022-01-01']

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(curve_filt['date_synth'], curve_filt['totalLiquidityUSD'] / 1e9,
        color='#E53935', linewidth=2.3, label='Curve Finance TVL (declining)')
ax.fill_between(curve_filt['date_synth'], curve_filt['totalLiquidityUSD'] / 1e9,
                alpha=0.1, color='#E53935')

ax.plot(rwa_agg_filt['date'], rwa_agg_filt['totalLiquidityUSD'] / 1e9,
        color='#2E7D32', linewidth=2.3, label='RWA Protocols Total TVL (growing)')
ax.fill_between(rwa_agg_filt['date'], rwa_agg_filt['totalLiquidityUSD'] / 1e9,
                alpha=0.1, color='#2E7D32')

ax.set_title('Fig 13: Curve TVL vs RWA Protocol TVL Growth (2022–2026)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('TVL (Billions USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30)
ax.legend(loc='upper left')
ax.text(0.02, 0.05,
        'Key finding: As Curve TVL declined due to over-reliance on incentives, RWA protocols grew steadily — diversification into RWA offers Pendle a path forward',
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', alpha=0.8))
fig.tight_layout()
plt.savefig(os.path.join(BASE, 'figures/fig13_curve_vs_rwa.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fig 13 saved.')

---
## Summary — Key Statistics

In [ ]:
# ── Summary: Print key verified statistics ─────────────────────────────────────
print('=' * 72)
print('PENDLE FINANCE TVL COLLAPSE — KEY STATISTICS SUMMARY')
print('=' * 72)

# ── Wave 1 ────────────────────────────────────────────────────────────────────
print('\n[WAVE 1 — Sep-25 Structural Expiry]')
print('-' * 40)

aave_reload = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/aave_pt_susde_sep2025.csv')
)
aave_reload['date'] = pd.to_datetime(aave_reload['date'])
pt_filt = aave_reload[aave_reload['asset'].isin(['PT-sUSDE-25SEP2025', 'PT-USDe-25SEP2025'])]
pt_peak = pt_filt.groupby('date')['totalAToken_units'].sum().max()
print(f'  PT Collateral Peak (Aave):          ${pt_peak/1e9:.3f}B  (Sep-24, 2025)')

susde_reload = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/susde_supply_aug_oct_2025.csv')
)
susde_reload['date'] = pd.to_datetime(susde_reload['date'])
susde_peak = susde_reload['sUSDe_supply'].max()
susde_peak_date = susde_reload.loc[susde_reload['sUSDe_supply'].idxmax(), 'date']
print(f'  sUSDe Supply Peak:                  ${susde_peak/1e9:.3f}B  ({susde_peak_date.strftime("%b-%d, %Y")})')

pool_reload = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave1_expiry/data/per_pool_tvl.csv')
)
pool_reload['day'] = pd.to_datetime(pool_reload['day'])
sep25_outflow = pool_reload[
    (pool_reload['pool_name'].isin(['PT-sUSDE-25SEP2025', 'PT-USDe-25SEP2025'])) &
    (pool_reload['day'] == '2025-09-25')
]['net_flow'].sum()
print(f'  Sep-25 Expiry Day Net Flow (combined): ${sep25_outflow/1e9:.3f}B')

# ── Wave 2 ────────────────────────────────────────────────────────────────────
print('\n[WAVE 2 — Trust Collapse Q4 2025]')
print('-' * 40)

yields_reload = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/pt_implied_yields_sep_dec_2025.csv')
)
yields_reload['date'] = pd.to_datetime(yields_reload['date'])
nov_yields_r = yields_reload[
    yields_reload['pool'].isin(['PT-sUSDE-27NOV2025', 'PT-USDe-27NOV2025']) &
    yields_reload['implied_APY_%'].notna()
]
print(f'  PT Implied APY Range (NOV pools):   {nov_yields_r["implied_APY_%"].min():.2f}% – {nov_yields_r["implied_APY_%"].max():.2f}%')

aave_borrow_r = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/aave_usdc_borrow_rate_sep_dec_2025.csv')
)
aave_rpc_r = aave_borrow_r[
    aave_borrow_r['source'] == 'Alchemy_RPC'
]['variableBorrowAPR_%'].dropna()
print(f'  Aave USDC Borrow Rate Range:        {aave_rpc_r.min():.2f}% – {aave_rpc_r.max():.2f}%')

vependle_r = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/vependle_lock_rate_oct_dec_2025.csv')
)
print(f'  vePENDLE Lock Rate Range:           {vependle_r["lock_rate_%"].min():.2f}% – {vependle_r["lock_rate_%"].max():.2f}%')

dau_q4_r = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave2_trust_collapse/data/dau_q4_2025.csv')
)
dau_q4_r['day'] = pd.to_datetime(dau_q4_r['day'])
dau_q4_filt = dau_q4_r[(dau_q4_r['day'] >= '2025-10-01') & (dau_q4_r['day'] <= '2025-12-31')]
dau_q4_peak_row = dau_q4_filt.loc[dau_q4_filt['dau'].idxmax()]
dau_q4_trough_row = dau_q4_filt.loc[dau_q4_filt['dau'].idxmin()]
print(f'  DAU Peak (Q4 2025):                 {int(dau_q4_peak_row["dau"]):,}  ({dau_q4_peak_row["day"].strftime("%b-%d")})')
print(f'  DAU Trough (Q4 2025):               {int(dau_q4_trough_row["dau"]):,}  ({dau_q4_trough_row["day"].strftime("%b-%d")})')

# ── Wave 3 ────────────────────────────────────────────────────────────────────
print('\n[WAVE 3 — Slow Bleed Q1 2026]')
print('-' * 40)

prices_r = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave3_slow_bleed/data/prices_jan_apr_2026.csv')
)
prices_r['date'] = pd.to_datetime(prices_r['date'])
prices_r = prices_r.sort_values('date')
nov1_r = prices_r[prices_r['date'] == '2025-11-01']
if nov1_r.empty:
    nov1_r = prices_r.iloc[[0]]
apr6_prices_r = prices_r[prices_r['date'] <= '2026-04-06'].iloc[-1]

for asset in ['BTC', 'ETH', 'PENDLE']:
    base_val = nov1_r[asset].values[0]
    end_val = apr6_prices_r[asset]
    chg_pct = (end_val - base_val) / base_val * 100
    print(f'  {asset} price change (Nov-1 → Apr-6):  {chg_pct:+.1f}%  (${end_val:.2f})')

gov_r = pd.read_csv(
    os.path.join(BASE, 'q2_metrics/token_governance/data/governance_stats.csv')
)
gov_r['date'] = pd.to_datetime(gov_r['date'])
gov_apr6 = gov_r[gov_r['date'] == '2026-04-06']
if not gov_apr6.empty and pd.notna(gov_apr6['sPENDLE_adoption_%'].values[0]):
    print(f'  sPENDLE Adoption at Apr-6:          {gov_apr6["sPENDLE_adoption_%"].values[0]:.2f}%')

dau_q1_r = pd.read_csv(
    os.path.join(BASE, 'q1_tvl_collapse/wave3_slow_bleed/data/dau_q1_2026.csv')
)
dau_q1_r['day'] = pd.to_datetime(dau_q1_r['day'])
dau_2026 = dau_q1_r[dau_q1_r['day'] >= '2026-01-01']
dau_floor_val = dau_2026[dau_2026['dau'] > 0]['dau'].min()
print(f'  DAU Floor (Q1 2026):                {int(dau_floor_val):,} users')
dau_apr6 = dau_q1_r[dau_q1_r['day'] == '2026-04-06']
if not dau_apr6.empty:
    print(f'  DAU at Apr-6:                       {int(dau_apr6["dau"].values[0]):,} users')

# ── Q2 ────────────────────────────────────────────────────────────────────────
print('\n[Q2 — User Behavior & Governance]')
print('-' * 40)

rollover_r = pd.read_csv(
    os.path.join(BASE, 'q2_metrics/user_behavior/data/pt_rollover_rate.csv')
)
for _, row in rollover_r.iterrows():
    print(f'  PT Rollover Rate ({row["expiry"]}):         {row["rollover_rate_pct"]:.1f}%')
    print(f'    Note: {row["note"]}')

yt_pt_r = pd.read_csv(
    os.path.join(BASE, 'q2_metrics/user_behavior/data/yt_pt_ratio.csv')
)
yt_pt_r['week'] = pd.to_datetime(yt_pt_r['week'])
pre_expiry = yt_pt_r[yt_pt_r['week'] <= '2025-09-22']['yt_pt_ratio'].mean()
post_expiry = yt_pt_r[yt_pt_r['week'] >= '2025-10-01']['yt_pt_ratio'].mean()
print(f'  YT/PT Ratio avg (pre Sep-25 expiry):   {pre_expiry:.3f}')
print(f'  YT/PT Ratio avg (post Sep-25 expiry):  {post_expiry:.3f}')

gov_latest = gov_r.sort_values('date').iloc[-1]
print(f'  vePENDLE Lock Rate (latest):        {gov_latest["vePENDLE_lock_rate_%"]:.2f}%  ({gov_latest["date"].strftime("%b-%d, %Y")})')
if pd.notna(gov_latest['sPENDLE_adoption_%']):
    print(f'  sPENDLE Adoption (latest):          {gov_latest["sPENDLE_adoption_%"]:.2f}%')

print('\n' + '=' * 72)
print('All figures saved to ./figures/')
print('=' * 72)